# SMILES 2026 Method 3 Logistic-Regularization Ablation

This notebook syncs the GitHub repo into a Drive-backed workspace, installs dependencies, and runs the dedicated Method 3 logistic-regression ablation on `data/dataset.csv` with `Qwen/Qwen2.5-0.5B`.

Study defaults:

- primary feature set: `logit_hidden`
- optional secondary replay: `logit`, `hidden`, `attns`, `logit_hidden_attns`
- full tokenized `prompt + response`
- truncation disabled
- float32 cache and float32 model forward pass
- batch size `1` for the cache build
- logistic families only: `L2`, `L1`, `Elastic-Net`


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)


In [ ]:
if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())


In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)


In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')


## Logistic-Regularization Ablation

This run keeps the feature extraction fixed and compares only logistic-regression regularization families. It writes `ablation_results.csv`, `ablation_results.json`, `best_config.json`, and `best_results.json` under `method3_llm_check_logreg_ablation/artifacts/ablation/`.


In [ ]:
run(
    'python method3_llm_check_logreg_ablation/run_ablation.py '
    '--batch-size 1 '
    '--secondary-feature-sets logit,hidden,attns,logit_hidden_attns '
    '--secondary-top-k 3',
    cwd=REPO_PATH,
)


In [ ]:
import json
import pandas as pd

ablation_dir = REPO_PATH / 'method3_llm_check_logreg_ablation' / 'artifacts' / 'ablation'
leaderboard = pd.read_csv(ablation_dir / 'ablation_results.csv')
display(
    leaderboard[
        [
            'name',
            'study_phase',
            'feature_set',
            'penalty',
            'solver',
            'C',
            'l1_ratio',
            'class_weight',
            'mean_val_accuracy',
            'mean_val_auroc',
            'mean_test_accuracy',
            'mean_test_auroc',
            'train_val_accuracy_gap',
            'train_val_auroc_gap',
        ]
    ]
)

best_payload = json.loads((ablation_dir / 'best_config.json').read_text())
print(json.dumps(best_payload, indent=2))


In [ ]:
combined_payload = json.loads((ablation_dir / 'ablation_results.json').read_text())
print('Combined JSON file:', ablation_dir / 'ablation_results.json')
print('Number of experiment rows:', len(combined_payload['experiments']))
